# Train the AI Act Risk-Tier Classifier on a free GPU (Colab or Kaggle T4)
Runtime > Change runtime type > **GPU (T4)**. Kaggle: Settings > Accelerator > GPU T4 x2, Internet on.
Upload `privify-aiact-tier-v0.1.zip` to the notebook files first. About 1.5 to 3 hours in total.

In [ ]:
!nvidia-smi

In [ ]:
import glob, os, shutil, subprocess
if not os.path.isdir("privify-aiact-tier"):
    pkgs = glob.glob("/kaggle/input/**/aiact_tier", recursive=True)      # Kaggle unzips uploads into a folder
    zips = glob.glob("/kaggle/input/**/privify-aiact-tier*.zip", recursive=True) + glob.glob("privify-aiact-tier*.zip") + glob.glob("/content/privify-aiact-tier*.zip")
    if pkgs:
        shutil.copytree(os.path.dirname(pkgs[0]), "privify-aiact-tier")
    elif zips:
        subprocess.run(["unzip", "-q", "-o", zips[0]], check=True)
    else:
        raise SystemExit("Repo not found. Add the zip as a Kaggle Dataset (Input panel > Upload) or upload it to Colab.")
print(os.listdir("privify-aiact-tier"))

In [ ]:
%cd privify-aiact-tier
!pip install -q -r requirements.txt bitsandbytes pytest
!python -m pytest -q

In [ ]:
!make data

In [ ]:
# 1. Quick sanity run (about 5 min): proves the GPU path works before the long run
!python -m aiact_tier.train_lora --load-4bit --batch-size 2 --grad-accum 8 --max-steps 20 --out outputs/sanity

In [ ]:
# 2. Full training run (1.5 to 3 h on a T4). Adapter saved to outputs/aiact-tier-lora
!make train-4bit

In [ ]:
# 3. Evaluate (torchao is removed first: Kaggle ships a version that breaks peft)
!pip uninstall -y -q torchao
!make eval-zeroshot
!make eval-ours

In [ ]:
# 4. Package only the small files (the full output folder holds ~800 MB of checkpoints)
!mkdir -p pkg && cp outputs/aiact-tier-lora/adapter_model.safetensors outputs/aiact-tier-lora/adapter_config.json outputs/aiact-tier-lora/train_meta.json pkg/
!cp -r eval/results pkg/results
!cd pkg && zip -qr ../results.zip . && ls -la ../results.zip